# Complete public task prefix vs instruction slice (twelve carriers), Kaggle T4

Question: does keeping the complete public task prefix help B use A's selected result states, compared with keeping only the instruction slice?

Same Tesla T4, same Qwen3-4B revision, FP16, greedy decoding, 384 plan / 48 answer tokens, same nine cases, frozen A outputs and result positions, same B prompts and framing. No A regeneration, no added reasons or calculations, no training. Development diagnostic on already-inspected cases, not confirmation.

Conditions (63 evaluations):
- **compact_baseline**: short prefix + generic instruction slice + newline + transferred R (should reproduce 63/108, 0/9)
- **task_prefix_r**: complete public task prefix (140/145/162 tokens) + newline + transferred R (candidate)
- **woodland_prefix_r**, **library_prefix_r**: frozen matched neutral prefixes + same instruction + newline + transferred R
- **task_prefix_fresh_r**: task prefix + newline + R token IDs processed again
- **full_kv**, **text_result**: references

**Cell 4** runs the check phase on all nine cases (no answers generated). Continue only if it passes.
**Cell 5** runs the first case, **Cell 6** the remaining eight (resumes), **Cell 7** prints the summary.

Upload `src/`, `configs/`, `scripts/`, `data/plan4_timing12.jsonl`, and the folders `results_stage_plan4_recon12/` and `results_stage_plan4_nbound/`.

Download `/kaggle/working/results_stage_plan4_publicprefix12`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
STAGE = "stage_plan4_publicprefix12"

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if f"{STAGE}.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / f"{STAGE}.py").is_file() and (cand / "data" / "plan4_timing12.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError(f"Need src/{STAGE}.py and data/plan4_timing12.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input") or not (dest / "src" / f"{STAGE}.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
for folder, names in (("results_stage_plan4_recon12", ("a_artifacts_12.jsonl", "evaluations.jsonl", "check_summary.json")),
                      ("results_stage_plan4_nbound", ("construction.json",))):
    for name in names:
        print(folder, name, (REPO / folder / name).is_file())

def out_file(name):
    for d in (WORK / "results_stage_plan4_publicprefix12", REPO / "results" / STAGE / "run"):
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(name)

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_publicprefix12. Check phase first.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--split", "plan4_timing12", "--phase", "check"])
print("check exit", rc)
cons = json.loads(out_file("construction.json").read_text(encoding="utf-8"))
check = json.loads(out_file("check_summary.json").read_text(encoding="utf-8"))

def logit_brief(res):
    return {k: (res.get(k) or {}).get("max_abs") for k in ("repeat", "ordinary_cached_vs_uncached", "completed_gather_vs_reencode")} | {"passed": res.get("passed")}

for t, block in cons["templates"].items():
    print(f"\n=== template {t} ok={block['ok']} (reuse continuation from {block['reuse_example']})")
    for ctx, row in block["contexts"].items():
        print(f"-- {ctx}: n_prefix={row['n_prefix']} private_hits={row['private_hits']} held={row['prefix_held']} "
              f"prepare={row['prepare_s']:.2f}s resident={row['resident_mib']:.2f} MiB")
        print("   reuse", logit_brief(row["prefix_reuse"]))
        if ctx == "task":
            b = row["boundary"]
            print("   prefill equals generic template:", row.get("prefill_equals_generic_template"))
            print("   boundary", {k: b[k] for k in ("prefix_matches_a", "ends_with_paragraph_break", "no_merge_across_boundary")})
            print("   last prefix tokens", b["tail_tokens"])
            print("   next A tokens (excluded)", b["next_a_tokens"])
        else:
            print("   matched", row["matched"])
        print("   decoded prefix:\n" + row["decoded_prefix"])

print("\n=== cases")
for c in check["cases"]:
    print(c["a_state_key"], "ok", c["ok"], "failed", c["failed"], "n_kv", c["n_kv"])
    print("   key shift vs baseline", round(c["keys_max_abs_vs_baseline"], 4), "| fresh vs transferred", c["fresh_vs_transferred"])
first = cons["cases"][0]
print("\nfresh reference (one-shot vs prefix then extension), first case:", logit_brief(first["fresh_reference"]))
r = first["renders"]["task_prefix_r"]
print("junction", r["junction"])
print("\nrendered task_prefix_r sequence (first case, with special tokens):\n" + r["prefix_text"] + "‖" + r["b_turn_text"])
print("\nall_passed", check["all_passed"], check["templates_ok"])
if not check["all_passed"]:
    raise RuntimeError("Construction check failed. Fix construction before generating any answers.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--split", "plan4_timing12", "--phase", "compare", "--limit", "1"])
print("first case exit", rc)
rows = [json.loads(l) for l in out_file("evaluations.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
for row in rows:
    print(row["example_id"], row["condition"], "labels", row.get("plan_n_status_correct"), "all12", row.get("plan_all_status_correct"),
          "I->E", row.get("plan_ineligible_marked_eligible"), "E->I", row.get("plan_eligible_marked_ineligible"),
          "choice", row.get("correct"), "n_kv", row.get("n_kv_positions"), "result_start", row.get("result_start"))

In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--split", "plan4_timing12", "--phase", "compare"])
print("compare exit", rc)

In [ ]:
data = json.loads(out_file("summary.json").read_text(encoding="utf-8"))
print("rows", data["n_rows"], "of", data["n_expected_rows"], "| cases", data["n_cases"], "| check", data["check"])
keys = ("labels_correct", "n_labels", "cases_all_twelve_correct", "ineligible_marked_eligible",
        "eligible_marked_ineligible", "parse_ok", "plan_truncated", "answer_truncated", "plan_lines_missing")
for c, block in data["labels"].items():
    print(c, {k: block.get(k) for k in keys})
print("\nfinal choice (separate)")
for c, block in data["final_choice"].items():
    print(c, block)
print("\nlabel changes vs compact baseline")
for c, tab in data["vs_baseline_label_changes"].items():
    print(c, tab)
print("\npaired by case (right minus left; 2000 bootstrap draws over the nine cases)")
for name, block in data["paired_by_case"].items():
    a, l = block["all_twelve_correct"], block["labels_correct"]
    print(name, "| all12", {k: a.get(k) for k in ("total_delta", "ci95_per_case", "cases_increased", "cases_decreased")},
          "| labels", {k: l.get(k) for k in ("total_delta", "ci95_per_case", "cases_increased", "cases_decreased")},
          "| fixed/broken", block["label_changes"].get("fixed"), block["label_changes"].get("broken"))
print("\nreproduction vs recon12", json.dumps(data["reproduction_vs_recon12"], indent=1))
print("meets recovery standard (9/9)", data["meets_recovery_standard"])
print("\ncosts")
for c, block in data["costs"].items():
    print(c, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in block.items()})